# PAW-Stimulated Bioethanol Production in *E. coli*
## A cross-model cross-chassis hybrid dFBA framework

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USER/REPO/blob/main/PAW_Ethanol_dFBA.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![DOI](https://img.shields.io/badge/DOI-pending-blue.svg)](#)

**Companion notebook for:** *"A cross-model cross-chassis hybrid dynamic flux balance analysis framework predicts a hormetic plasma-activated water operating window for ethanol production in* Escherichia coli*"*

---

## Overview

This notebook implements the full computational pipeline described in Sections 2.1–2.12 of the manuscript:

1. **PAW chemistry** — first-order RONS decay (H₂O₂, NO₂⁻, NO₃⁻, ONOO⁻) → stress index *S(t)* (§2.2)
2. **Redox-partitioning controller** — bell-shaped *r*(*S*) that couples ethanol/formate exchange flux ratio (§2.3, central contribution)
3. **Cell death + ATPM burden** — Hill-form kinetics + linear stress-cost augmentation (§2.4)
4. **GSMM + chassis** — iJO1366 / iML1515 × {WT, KO11, LY01} (§2.5)
5. **dFBA integration** — 10-h batch with Δt = 0.25 h (§2.6)
6. **Analyses 3.1–3.8** — baseline → dose-response → mechanistic decomposition → scenarios → carbon yield → Monte Carlo (N = 1,800) → Morris → literature cross-validation

## Key results (reproduced by this notebook)

| Chassis | iJO1366 baseline | iJO1366 peak | iML1515 baseline | iML1515 peak | α* |
|---------|------------------|--------------|------------------|--------------|----|
| WT      | 17.0 mM          | 20.0 mM (+18%) | 15.9 mM        | 20.3 mM (+28%) | 0.60 |
| KO11    | 18.0 mM          | 20.0 mM        | 16.5 mM        | 20.5 mM        | 0.60 |
| LY01    | 25.2 mM          | 25.2 mM (no response) | 23.5 mM | 23.5 mM (no response) | — |

**Central biological message:** PAW acts as a *non-genetic functional analogue* of the Δ*pta–ackA* deletion — it produces in WT/KO11 the phenotype that LY01 achieves through genetic engineering.

## Runtime

~ 30–40 min on a standard Colab CPU runtime (no GPU required). All linear programmes solved with GLPK 5.0 through COBRApy 0.31.

## How to cite

If you use this code or framework, please cite the accompanying manuscript:

> [Author names]. (2026). A cross-model cross-chassis hybrid dynamic flux balance analysis framework predicts a hormetic plasma-activated water operating window for ethanol production in *Escherichia coli*. *Biotechnology and Bioengineering*, [in press].

---

## 1. Environment setup

Install dependencies and verify versions. All packages pin via `pip` for reproducibility.

In [ ]:
# Install dependencies (Colab-friendly versions)
!pip install cobra==0.29.1 pandas numpy matplotlib scipy openpyxl tqdm -q

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import cobra
from cobra.io import load_model
import time
from tqdm import tqdm
import pickle
import os

print(f"COBRApy version: {cobra.__version__}")
print(f"NumPy version:   {np.__version__}")
print(f"Pandas version:  {pd.__version__}")
print(f"Matplotlib:      {plt.matplotlib.__version__}")

### 1.1. Mount Google Drive for persistent storage (optional)

Skip this cell if running outside Colab. Output files (CSV, Excel, figures) will then be saved to the local notebook directory.

In [ ]:
# Optional: mount Drive for persistent storage
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUTPUT_DIR = '/content/drive/MyDrive/PAW_Ethanol_dFBA/'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Output directory: {OUTPUT_DIR}")
except (ImportError, ModuleNotFoundError):
    OUTPUT_DIR = './PAW_Ethanol_outputs/'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Output directory (local): {OUTPUT_DIR}")

---

## 2. Load genome-scale metabolic reconstructions (§2.5)

Two reconstructions of *E. coli* K-12 MG1655:

- **iJO1366** (Orth et al., 2011): 2,583 reactions, 1,805 metabolites, 1,367 genes
- **iML1515** (Monk et al., 2017): 2,712 reactions, 1,877 metabolites, 1,515 genes

Both configured for anaerobic batch: O₂ exchange lower bound = 0, glucose exchange lower bound = −10 mmol gDW⁻¹ h⁻¹.

In [ ]:
print("Loading iJO1366...")
m_J = load_model("iJO1366")
print(f"  -> {len(m_J.reactions)} reactions, {len(m_J.metabolites)} metabolites, {len(m_J.genes)} genes")

print("Loading iML1515...")
m_M = load_model("iML1515")
print(f"  -> {len(m_M.reactions)} reactions, {len(m_M.metabolites)} metabolites, {len(m_M.genes)} genes")

# Anaerobic configuration
for m in (m_J, m_M):
    m.reactions.EX_o2_e.lower_bound = 0.0
    m.reactions.EX_glc__D_e.lower_bound = -10.0

print("\nBoth models configured for anaerobic batch.")

---

## 3. Model parameters (Table 1 of manuscript)

All parameters centralized in a single class for reproducibility. Monte Carlo ranges as inline comments.

In [ ]:
class P:
    """Central parameter container. Values match Table 1 of the manuscript."""

    # --- PAW chemistry (§2.2; Lukes 2014, Tarabova 2018) ---
    H2O2_0     = 5.0    # mM at alpha=1
    NO2_0      = 2.5
    NO3_0      = 1.0
    ONOO_0     = 0.5

    k_H2O2     = 0.277  # h^-1 (t_half = 2.5 h)
    k_NO2      = 0.150
    k_NO3      = 0.020
    k_ONOO     = 1.500

    w_H2O2     = 1.0    # Stress-equivalent weights (Imlay 2008)
    w_NO2      = 0.4
    w_NO3      = 0.05
    w_ONOO     = 5.0

    # --- Cell death + ATPM (§2.4) ---
    kd_max     = 4.0    # h^-1                 MC: [2.5, 6.0]
    S_half_d   = 14.0   # mM-eq
    n_d        = 4
    beta_ATPM  = 0.10   # mmol gDW^-1 h^-1 (mM-eq)^-1  MC: [0.06, 0.14]

    # --- Redox-partitioning controller (§2.3 — central contribution) ---
    r_base     = 0.5    # baseline ethanol/formate ratio (non-binding at S=0)
    r_max      = 10.0   # maximum forced ratio   MC: [6, 14]
    Ka_r       = 1.8    # mM-eq (activation)      MC: [0.8, 3.0]
    n_a        = 2
    Ki_r       = 13.0   # mM-eq (damping)         MC: [9, 16]
    n_i        = 4

    # --- PFL inhibition (Knappe & Sawers 1990; Wagner 1992) ---
    PFL_inh_K  = 0.8    # mM-eq                   MC: [0.4, 1.5]
    PFL_inh_n  = 3

    # --- NADH detoxification burden ---
    gamma_NADH = 0.02   # mmol gDW^-1 h^-1 (mM-eq)^-1  MC: [0.01, 0.04]

    # --- Engineered chassis ---
    ADH_eng    = -40.0  # ALCD2x lower bound for KO11/LY01 (pdc-adhB-cat cassette)

    # --- Integration (§2.6) ---
    dt         = 0.25   # h
    T_batch    = 10.0   # h (40 steps)

    # --- Initial conditions ---
    glc_0      = 55.0   # mM
    X_0        = 0.05   # gDW/L
    glc_uptake_max = 10.0  # mmol gDW^-1 h^-1

print("Parameters loaded.")
print(f"  alpha_PAW design range: [0, 1.5]")
print(f"  Batch horizon: {P.T_batch} h, step {P.dt} h ({int(P.T_batch/P.dt)} steps)")
print(f"  Initial: glucose = {P.glc_0} mM, biomass = {P.X_0} gDW/L")

---

## 4. Core simulation functions

Six functions implement the framework:

| Function | Equation | Section |
|----------|----------|---------|
| `paw_species(t, alpha)` | Eq. (1): analytical RONS decay | §2.2 |
| `stress_S(t, alpha)` | Eq. (2): hydrogen-peroxide-equivalent stress index | §2.2 |
| `r_func(S)` | Eq. (3): bell-shaped controller | §2.3 |
| `pfl_inhibition(S)` | Eq. (5): PFL multiplicative reduction | §2.3 |
| `kd_func(S)` | Eq. (6): Hill-form cell death kernel | §2.4 |
| `atpm_burden(S)` | Eq. (7): ATPM augmentation | §2.4 |

In [ ]:
def paw_species(t, alpha):
    """Eq. (1): Analytical first-order decay of four RONS species."""
    H2O2 = alpha * P.H2O2_0 * np.exp(-P.k_H2O2 * t)
    NO2  = alpha * P.NO2_0  * np.exp(-P.k_NO2  * t)
    NO3  = alpha * P.NO3_0  * np.exp(-P.k_NO3  * t)
    ONOO = alpha * P.ONOO_0 * np.exp(-P.k_ONOO * t)
    return H2O2, NO2, NO3, ONOO

def stress_S(t, alpha):
    """Eq. (2): Aggregate hydrogen-peroxide-equivalent stress index."""
    H2O2, NO2, NO3, ONOO = paw_species(t, alpha)
    return P.w_H2O2*H2O2 + P.w_NO2*NO2 + P.w_NO3*NO3 + P.w_ONOO*ONOO

def r_func(S):
    """Eq. (3): Bell-shaped redox-partitioning controller r(S)."""
    activation = (S**P.n_a) / (P.Ka_r**P.n_a + S**P.n_a)
    damping = 1.0 / (1.0 + (S / P.Ki_r)**P.n_i)
    return P.r_base + (P.r_max - P.r_base) * activation * damping

def pfl_inhibition(S):
    """Eq. (5): Multiplicative reduction on PFL upper bound."""
    return 1.0 / (1.0 + (S / P.PFL_inh_K)**P.PFL_inh_n)

def kd_func(S):
    """Eq. (6): Hill-form specific cell death rate."""
    return P.kd_max * (S**P.n_d) / (P.S_half_d**P.n_d + S**P.n_d)

def atpm_burden(S):
    """Eq. (7): Linear augmentation of ATP maintenance flux."""
    return P.beta_ATPM * S + P.gamma_NADH * S

print("Function sanity checks at S = 0, 2, 6, 14 mM-eq:")
print(f"{'S':>6} {'r(S)':>8} {'PFL_inh':>10} {'k_d':>8} {'ATPM':>8}")
for S in [0, 2, 6, 14]:
    print(f"{S:>6.1f} {r_func(S):>8.3f} {pfl_inhibition(S):>10.3f} {kd_func(S):>8.3f} {atpm_burden(S):>8.3f}")

---

## 5. Chassis configuration (§2.5)

| Chassis | Genotype | Implementation |
|---------|----------|----------------|
| **WT**   | Wild-type *E. coli* K-12 | Default reconstruction |
| **KO11** | Δ*frdABCD*, Δ*ldhA*, *pdc-adhB-cat*⁺ | FRD2=0, FRD3=0, LDH_D=0, ALCD2x lb = −40 |
| **LY01** | KO11 + Δ*pta–ackA* | KO11 + ACKr=0 |

In [ ]:
def get_biomass_rxn(model):
    for r in model.reactions:
        if 'BIOMASS' in r.id.upper() and 'core' in r.id.lower():
            return r.id
    for r in model.reactions:
        if 'BIOMASS' in r.id.upper():
            return r.id
    raise ValueError("No biomass reaction found.")

def configure_chassis(model, chassis):
    m = model.copy()
    bio_rxn = get_biomass_rxn(m)
    m.objective = bio_rxn

    m.reactions.EX_o2_e.lower_bound = 0.0
    m.reactions.EX_glc__D_e.lower_bound = -P.glc_uptake_max

    if chassis == 'WT':
        pass
    elif chassis == 'KO11':
        for rid in ['FRD2', 'FRD3']:
            if rid in m.reactions:
                m.reactions.get_by_id(rid).bounds = (0, 0)
        if 'LDH_D' in m.reactions:
            m.reactions.LDH_D.bounds = (0, 0)
        m.reactions.ALCD2x.lower_bound = P.ADH_eng
    elif chassis == 'LY01':
        for rid in ['FRD2', 'FRD3']:
            if rid in m.reactions:
                m.reactions.get_by_id(rid).bounds = (0, 0)
        if 'LDH_D' in m.reactions:
            m.reactions.LDH_D.bounds = (0, 0)
        m.reactions.ALCD2x.lower_bound = P.ADH_eng
        if 'ACKr' in m.reactions:
            m.reactions.ACKr.bounds = (0, 0)
    else:
        raise ValueError(f"Unknown chassis: {chassis}")

    coupling_constr = m.problem.Constraint(
        m.reactions.EX_etoh_e.flux_expression - P.r_base * m.reactions.EX_for_e.flux_expression,
        lb=0, ub=None, name='redox_coupling'
    )
    m.add_cons_vars([coupling_constr])
    return m, bio_rxn

# Verify on iJO1366
for chassis in ['WT', 'KO11', 'LY01']:
    m_test, bio = configure_chassis(m_J, chassis)
    sol = m_test.optimize()
    print(f"{chassis:6s}: mu = {sol.objective_value:.4f} h^-1, "
          f"EtOH = {-sol.fluxes['EX_etoh_e']:.2f}, "
          f"Acetate = {-sol.fluxes['EX_ac_e']:.2f} mmol gDW^-1 h^-1")

---

## 6. dFBA integration scheme (§2.6)

Static-optimization approach (Mahadevan et al., 2002). The coupling coefficient is updated via `set_linear_coefficients` on the existing optlang Constraint (8× speed-up vs. remove/re-add).

In [ ]:
def update_redox_coupling(m, r_val):
    constr = m.constraints['redox_coupling']
    constr.set_linear_coefficients({
        m.reactions.EX_etoh_e.forward_variable: 1.0,
        m.reactions.EX_etoh_e.reverse_variable: -1.0,
        m.reactions.EX_for_e.forward_variable:  -r_val,
        m.reactions.EX_for_e.reverse_variable:   r_val,
    })

def run_dfba(model, chassis, alpha, T=None, dt=None,
             scenario='full', verbose=False):
    """Run dFBA for one chassis-model combination at given alpha.

    Scenarios: 'full', 'baseline', 'coupling', 'pfl', 'no_alcd'
    """
    if T is None: T = P.T_batch
    if dt is None: dt = P.dt
    n_steps = int(T / dt)

    m, bio_rxn = configure_chassis(model, chassis)
    if scenario == 'no_alcd':
        m.reactions.ALCD2x.bounds = (0, 0)

    pfl_baseline = m.reactions.PFL.upper_bound
    atpm_baseline = m.reactions.ATPM.lower_bound

    t_arr = np.zeros(n_steps + 1)
    X_arr = np.zeros(n_steps + 1)
    S_arr = np.zeros(n_steps + 1)
    glc_arr = np.zeros(n_steps + 1)
    etoh_arr = np.zeros(n_steps + 1)
    for_arr = np.zeros(n_steps + 1)
    ac_arr = np.zeros(n_steps + 1)

    t_arr[0] = 0
    X_arr[0] = P.X_0
    glc_arr[0] = P.glc_0
    S_arr[0] = stress_S(0, alpha)

    for i in range(n_steps):
        t = i * dt
        S_now = stress_S(t, alpha)
        S_arr[i] = S_now

        if scenario == 'baseline':
            r_val = P.r_base
            pfl_factor = 1.0
        elif scenario == 'coupling':
            r_val = r_func(S_now)
            pfl_factor = 1.0
        elif scenario == 'pfl':
            r_val = P.r_base
            pfl_factor = pfl_inhibition(S_now)
        else:
            r_val = r_func(S_now)
            pfl_factor = pfl_inhibition(S_now)

        update_redox_coupling(m, r_val)
        m.reactions.PFL.upper_bound = pfl_baseline * pfl_factor
        m.reactions.ATPM.lower_bound = atpm_baseline + atpm_burden(S_now)

        if glc_arr[i] < 0.1:
            m.reactions.EX_glc__D_e.lower_bound = 0.0
        else:
            m.reactions.EX_glc__D_e.lower_bound = -P.glc_uptake_max

        try:
            sol = m.optimize()
            if sol.status != 'optimal':
                X_arr[i+1] = X_arr[i] * np.exp(-kd_func(S_now) * dt)
                glc_arr[i+1] = glc_arr[i]; etoh_arr[i+1] = etoh_arr[i]
                for_arr[i+1] = for_arr[i]; ac_arr[i+1] = ac_arr[i]
                t_arr[i+1] = t + dt
                continue

            mu = sol.objective_value
            v_glc = sol.fluxes['EX_glc__D_e']
            v_etoh = sol.fluxes['EX_etoh_e']
            v_for = sol.fluxes['EX_for_e']
            v_ac = sol.fluxes['EX_ac_e']

            kd = kd_func(S_now)
            X_arr[i+1] = X_arr[i] * np.exp((mu - kd) * dt)
            glc_arr[i+1] = max(0, glc_arr[i] + v_glc * X_arr[i] * dt)
            etoh_arr[i+1] = etoh_arr[i] + v_etoh * X_arr[i] * dt
            for_arr[i+1] = for_arr[i] + v_for * X_arr[i] * dt
            ac_arr[i+1] = ac_arr[i] + v_ac * X_arr[i] * dt
            t_arr[i+1] = t + dt
        except Exception as e:
            if verbose:
                print(f"  Step {i} failed: {e}")
            X_arr[i+1] = X_arr[i]; glc_arr[i+1] = glc_arr[i]
            etoh_arr[i+1] = etoh_arr[i]; for_arr[i+1] = for_arr[i]
            ac_arr[i+1] = ac_arr[i]; t_arr[i+1] = t + dt

    S_arr[-1] = stress_S(T, alpha)
    glc_consumed = P.glc_0 - glc_arr[-1]
    yield_etoh = etoh_arr[-1] / glc_consumed if glc_consumed > 0 else 0

    return {
        't': t_arr, 'X': X_arr, 'S': S_arr,
        'glc': glc_arr, 'etoh': etoh_arr, 'for': for_arr, 'ac': ac_arr,
        'EtOH_final': etoh_arr[-1], 'For_final': for_arr[-1],
        'Ac_final': ac_arr[-1], 'X_final': X_arr[-1],
        'peak_S': S_arr.max(), 'yield': yield_etoh,
    }

print("dFBA integrator ready.")

---

## 7. Analysis 3.1 — Cross-model baseline (no PAW exposure)

Reproduces **Table 2** of the manuscript.

In [ ]:
print("=" * 82)
print("Section 3.1 - Cross-model baseline (alpha = 0)")
print("=" * 82)

baseline_results = []
for chassis in ['WT', 'KO11', 'LY01']:
    for gsmm_name, m in [('iJO1366', m_J), ('iML1515', m_M)]:
        res = run_dfba(m, chassis, alpha=0.0)
        baseline_results.append({
            'chassis': chassis, 'gsmm': gsmm_name,
            'EtOH (mM)': res['EtOH_final'],
            'Acetate (mM)': res['Ac_final'],
            'Yield (mol/mol)': res['yield'],
            'Biomass (gDW/L)': res['X_final'],
        })
        print(f"  {chassis:6s} / {gsmm_name:8s}: "
              f"EtOH={res['EtOH_final']:5.1f}  "
              f"Ac={res['Ac_final']:5.1f}  "
              f"Y={res['yield']:5.2f}  "
              f"X={res['X_final']:5.2f}")

df_baseline = pd.DataFrame(baseline_results)
df_baseline.to_csv(os.path.join(OUTPUT_DIR, 'table2_baseline.csv'), index=False)
print(f"\nSaved: {OUTPUT_DIR}table2_baseline.csv")

---

## 8. Analysis 3.2 — Hormetic dose-response sweep

16 α values × 3 chassis × 2 GSMM = **96 simulations**. Reproduces **Figure 1**: hormetic peak at α* = 0.60.

In [ ]:
alphas = np.linspace(0, 1.5, 16)
sweep_results = []

print(f"Running {len(alphas)} alpha-values x 3 chassis x 2 GSMM = {len(alphas)*6} simulations...\n")

for chassis in ['WT', 'KO11', 'LY01']:
    for gsmm_name, m in [('iJO1366', m_J), ('iML1515', m_M)]:
        for alpha in tqdm(alphas, desc=f"{chassis}/{gsmm_name}", leave=False):
            res = run_dfba(m, chassis, alpha)
            sweep_results.append({
                'chassis': chassis, 'gsmm': gsmm_name, 'alpha': alpha,
                'EtOH': res['EtOH_final'], 'For': res['For_final'],
                'Ac': res['Ac_final'], 'X': res['X_final'],
                'peak_S': res['peak_S'], 'yield': res['yield'],
            })

df_sweep = pd.DataFrame(sweep_results)
df_sweep.to_csv(os.path.join(OUTPUT_DIR, 'figure1_dose_response_sweep.csv'), index=False)
print(f"\nSaved: figure1_dose_response_sweep.csv ({len(df_sweep)} rows)")

In [ ]:
# ===== Figure 1: Cross-chassis hormetic dose-response =====
COL = {'WT': '#1f77b4', 'KO11': '#2ca02c', 'LY01': '#ff7f0e'}

fig, ax = plt.subplots(figsize=(8, 5.5), facecolor='white')

for chassis in ['WT', 'KO11', 'LY01']:
    sub_J = df_sweep[(df_sweep['gsmm']=='iJO1366') & (df_sweep['chassis']==chassis)].sort_values('alpha')
    sub_M = df_sweep[(df_sweep['gsmm']=='iML1515') & (df_sweep['chassis']==chassis)].sort_values('alpha')

    ax.plot(sub_J['alpha'], sub_J['EtOH'], '-o', color=COL[chassis], lw=2.2, ms=5.5,
            label=f'{chassis} / iJO1366', alpha=1.0, markeredgecolor='white', markeredgewidth=0.6)
    ax.plot(sub_M['alpha'], sub_M['EtOH'], '--s', color=COL[chassis], lw=1.8, ms=4.8,
            alpha=0.6, markeredgecolor='white', markeredgewidth=0.5,
            label=f'{chassis} / iML1515')

ax.axvspan(0.4, 0.7, alpha=0.10, color='#ffd700', zorder=0)
ax.text(0.55, 28.5, 'Operating window', ha='center', fontsize=9.5,
        color='#806000', fontweight='bold')

sub_wt = df_sweep[(df_sweep['gsmm']=='iJO1366') & (df_sweep['chassis']=='WT')]
peak_row = sub_wt.sort_values('EtOH', ascending=False).iloc[0]
ax.scatter([peak_row['alpha']], [peak_row['EtOH']], s=260, marker='*',
           color='#ffd700', edgecolor='#333', linewidth=1.3, zorder=10)

ax.set_xlabel(r'Dimensionless PAW dose ($\alpha_{PAW}$)', fontsize=11)
ax.set_ylabel('Final ethanol titer (mM)', fontsize=11)
ax.legend(fontsize=8.5, loc='lower left', ncol=2, framealpha=0.95, edgecolor='#666')
ax.grid(True, alpha=0.3, linestyle='--', lw=0.5)
ax.set_xlim(-0.05, 1.55)
ax.set_ylim(0, 30)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'Figure_1_cross_chassis_dose_response.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: Figure_1_cross_chassis_dose_response.png")

---

## 9. Figure 2 — Mechanistic decomposition of the controller (§3.3)

In [ ]:
# ===== Figure 2: Controller decomposition =====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 5), facecolor='white')

S_range = np.linspace(0, 20, 300)
r_vals = np.array([r_func(s) for s in S_range])
pfl_vals = np.array([pfl_inhibition(s) for s in S_range])
growth_vals = np.array([1.0 - kd_func(s)/P.kd_max for s in S_range])

ax1.plot(S_range, r_vals, color='#1f77b4', lw=2.5, label='Redox coupling $r(S)$')
ax1.plot(S_range, pfl_vals * P.r_max, color='#d62728', lw=2.5, label='PFL activity x $r_{max}$')
ax1.plot(S_range, growth_vals * P.r_max, color='#ff7f0e', lw=2.5, ls='--',
         label='Growth retention x $r_{max}$')

peak_idx = np.argmax(r_vals)
ax1.axvline(S_range[peak_idx], color='#2ca02c', ls=':', lw=1.5, alpha=0.7)
ax1.scatter([S_range[peak_idx]], [r_vals[peak_idx]], s=120, marker='*',
            color='#ffd700', edgecolor='#333', linewidth=1, zorder=10)

ax1.set_title('(a) Component drivers of M(S)', fontsize=11, fontweight='bold', loc='left', pad=8)
ax1.set_xlabel('Stress index $S$ (mM-eq)', fontsize=10)
ax1.set_ylabel('Driver value', fontsize=10)
ax1.legend(fontsize=9, loc='upper right', framealpha=0.92, edgecolor='#333')
ax1.grid(True, alpha=0.3, linestyle='--', lw=0.5)
ax1.set_xlim(0, 20)

alphas_b = np.linspace(0, 1.5, 100)
S_at_alpha = alphas_b * (P.w_H2O2*P.H2O2_0 + P.w_NO2*P.NO2_0 + P.w_NO3*P.NO3_0 + P.w_ONOO*P.ONOO_0)
r_at_alpha = np.array([r_func(s) for s in S_at_alpha])

ax2.plot(alphas_b, r_at_alpha, color='#1f77b4', lw=2.5)
ax2.fill_between(alphas_b, P.r_base, r_at_alpha, alpha=0.15, color='#1f77b4')
ax2.axvspan(0.4, 0.7, alpha=0.12, color='#ffd700')
ax2.text(0.55, 5.5, 'Operating\nwindow', ha='center', fontsize=9, color='#806000', fontweight='bold')
ax2.axhline(P.r_base, color='#666', ls=':', lw=1, alpha=0.7)

ax2.set_title('(b) r(S) projected onto PAW dose', fontsize=11, fontweight='bold', loc='left', pad=8)
ax2.set_xlabel(r'Dimensionless PAW dose ($\alpha_{PAW}$)', fontsize=10)
ax2.set_ylabel(r'Forced flux ratio $r(\alpha_{PAW})$', fontsize=10)
ax2.grid(True, alpha=0.3, linestyle='--', lw=0.5)
ax2.set_xlim(-0.05, 1.55)
ax2.set_ylim(0, 11)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'Figure_2_controller_decomposition.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: Figure_2_controller_decomposition.png")

---

## 10. Analysis 3.4 — Scenario decomposition (§3.4)

Reproduces **Table 3** + synergy diagnostic.

In [ ]:
scenarios = ['baseline', 'coupling', 'pfl', 'full', 'no_alcd']
labels = {
    'baseline': '(A) Baseline (no PAW)',
    'coupling': '(B) Coupling only',
    'pfl':      '(C) PFL inhibition only',
    'full':     '(D) Full PAW (operating)',
    'no_alcd':  '(E) D with ALCD2x=0 (control)',
}

print("Scenario decomposition (WT/iJO1366 at alpha=0.60):")
print("=" * 70)

scenario_results = {}
for sc in scenarios:
    alpha = 0.0 if sc == 'baseline' else 0.6
    res = run_dfba(m_J, 'WT', alpha, scenario=sc)
    scenario_results[sc] = res
    print(f"  {labels[sc]:40s} -> EtOH = {res['EtOH_final']:5.2f} mM")

vA = scenario_results['baseline']['EtOH_final']
vB = scenario_results['coupling']['EtOH_final']
vC = scenario_results['pfl']['EtOH_final']
vD = scenario_results['full']['EtOH_final']
S_syn = vD / (vB + vC - vA) if (vB + vC - vA) > 0 else float('nan')
print(f"\nSynergy ratio S_syn = v_D / (v_B + v_C - v_A) = {S_syn:.3f}")

df_scenarios = pd.DataFrame([
    {'scenario': labels[sc], 'EtOH_final_mM': scenario_results[sc]['EtOH_final']}
    for sc in scenarios
])
df_scenarios.to_csv(os.path.join(OUTPUT_DIR, 'table3_scenarios.csv'), index=False)

---

## 11. Figure 3 — Carbon redistribution and yield (§3.5)

In [ ]:
# ===== Figure 3: Carbon redistribution and yield =====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 5), facecolor='white')

for chassis in ['WT', 'KO11']:
    sub = df_sweep[(df_sweep['gsmm']=='iJO1366') & (df_sweep['chassis']==chassis)].sort_values('alpha')
    ax1.plot(sub['alpha'], sub['Ac'], '-o', color=COL[chassis], lw=2, ms=5,
             markeredgecolor='white', markeredgewidth=0.5, label=f'Acetate ({chassis})')
    ax1.plot(sub['alpha'], sub['For'], '--^', color=COL[chassis], lw=1.5, ms=5,
             alpha=0.7, markeredgecolor='white', markeredgewidth=0.5,
             label=f'Formate ({chassis})')

ax1.set_title('(a) By-product redistribution', fontsize=11, fontweight='bold', loc='left', pad=8)
ax1.set_xlabel(r'Dimensionless PAW dose ($\alpha_{PAW}$)', fontsize=10)
ax1.set_ylabel('Cumulative by-product titer (mM)', fontsize=10)
ax1.legend(fontsize=9, loc='upper right', framealpha=0.92, edgecolor='#333')
ax1.grid(True, alpha=0.3, linestyle='--', lw=0.5)
ax1.set_xlim(-0.05, 1.55)

for chassis in ['WT', 'KO11', 'LY01']:
    sub = df_sweep[(df_sweep['gsmm']=='iJO1366') & (df_sweep['chassis']==chassis)].sort_values('alpha')
    ax2.plot(sub['alpha'], sub['yield'], '-o', color=COL[chassis], lw=2, ms=5,
             markeredgecolor='white', markeredgewidth=0.5, label=chassis)

ax2.axhline(2.0, color='#666', ls=':', lw=1.2)
ax2.text(0.02, 2.05, '  Stoichiometric maximum (2.0 mol/mol)', fontsize=8.5, color='#444', va='bottom')
ax2.axhline(1.65, color='#ff7f0e', ls=':', lw=1, alpha=0.7)
ax2.text(0.02, 1.65, '  LY01 baseline (1.65 mol/mol)', fontsize=8.5, color='#ff7f0e', va='bottom')

ax2.set_title('(b) Ethanol yield', fontsize=11, fontweight='bold', loc='left', pad=8)
ax2.set_xlabel(r'Dimensionless PAW dose ($\alpha_{PAW}$)', fontsize=10)
ax2.set_ylabel('Ethanol yield (mol EtOH / mol glucose)', fontsize=10)
ax2.legend(fontsize=9.5, loc='lower right', framealpha=0.92, edgecolor='#333')
ax2.grid(True, alpha=0.3, linestyle='--', lw=0.5)
ax2.set_xlim(-0.05, 1.55)
ax2.set_ylim(0.5, 2.2)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'Figure_3_carbon_redistribution_yield.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: Figure_3_carbon_redistribution_yield.png")

---

## 12. Analysis 3.6 — Monte Carlo robustness (§3.6)

Latin hypercube sampling: **N = 300 per chassis × 1,800 simulations + 100 enrichment per condition** in [0.5, 0.7].
Takes ~15-25 min on Colab CPU.

In [ ]:
mc_ranges = {
    'alpha':     (0.0, 1.5),
    'r_max':     (6.0, 14.0),
    'kd_max':    (2.5, 6.0),
    'beta_ATPM': (0.06, 0.14),
    'Ka_r':      (0.8, 3.0),
    'Ki_r':      (9.0, 16.0),
    'PFL_inh_K': (0.4, 1.5),
    'gamma_NADH':(0.01, 0.04),
}

def latin_hypercube(n_samples, n_dims, seed=42):
    rng = np.random.default_rng(seed)
    samples = np.zeros((n_samples, n_dims))
    for d in range(n_dims):
        perm = rng.permutation(n_samples)
        samples[:, d] = (perm + rng.uniform(0, 1, n_samples)) / n_samples
    return samples

def mc_run(chassis, model, n=300, n_enrichment=100, seed=42):
    n_dims = len(mc_ranges)
    lhs = latin_hypercube(n, n_dims, seed)

    lhs_enrich = latin_hypercube(n_enrichment, n_dims, seed + 1)
    lhs_enrich[:, 0] = (lhs_enrich[:, 0] * (0.7 - 0.5) + 0.5) / 1.5
    lhs_all = np.vstack([lhs, lhs_enrich])

    keys = list(mc_ranges.keys())
    results = []

    for s in tqdm(lhs_all, desc=f'MC {chassis}', leave=False):
        params = {}
        for k, val_norm in zip(keys, s):
            lo, hi = mc_ranges[k]
            params[k] = lo + val_norm * (hi - lo)

        orig = {k: getattr(P, k) for k in keys if k != 'alpha'}
        for k, v in params.items():
            if k != 'alpha':
                setattr(P, k, v)

        try:
            res = run_dfba(model, chassis, params['alpha'])
            row = {**params, 'EtOH': res['EtOH_final'], 'peak_S': res['peak_S'],
                   'X_final': res['X_final'], 'yield': res['yield']}
            results.append(row)
        except Exception:
            pass
        finally:
            for k, v in orig.items():
                setattr(P, k, v)

    return pd.DataFrame(results)

print("Running Monte Carlo (iJO1366, all chassis)...")
mc_dfs = []
for chassis in ['WT', 'KO11', 'LY01']:
    df_mc = mc_run(chassis, m_J, n=300, n_enrichment=100, seed=42)
    df_mc['chassis'] = chassis
    df_mc['gsmm'] = 'iJO1366'
    mc_dfs.append(df_mc)

df_mc_full = pd.concat(mc_dfs, ignore_index=True)
df_mc_full.to_csv(os.path.join(OUTPUT_DIR, 'figure4_monte_carlo.csv'), index=False)
print(f"\nTotal Monte Carlo samples: {len(df_mc_full)}")

In [ ]:
# Operating-window statistics
print("Operating-window robustness (alpha in [0.5, 0.7]):")
print("=" * 80)
print(f"{'Chassis':10s} {'N':>5s} {'Mean':>7s} {'SD':>7s} {'CV (%)':>8s} {'P5':>6s} {'P95':>6s} {'Safe%':>7s} {'Viab%':>7s}")
print("-" * 80)
for chassis in ['WT', 'KO11', 'LY01']:
    sub = df_mc_full[(df_mc_full['chassis']==chassis) &
                     (df_mc_full['alpha']>=0.5) &
                     (df_mc_full['alpha']<=0.7)]
    if len(sub) > 5:
        cv = (sub['EtOH'].std() / sub['EtOH'].mean()) * 100
        safe_pct = (sub['peak_S'] < 10.0).mean() * 100
        viab_pct = (sub['X_final'] > 0.1).mean() * 100
        print(f"{chassis:10s} {len(sub):>5d} {sub['EtOH'].mean():>7.2f} "
              f"{sub['EtOH'].std():>7.2f} {cv:>8.2f} "
              f"{sub['EtOH'].quantile(0.05):>6.2f} {sub['EtOH'].quantile(0.95):>6.2f} "
              f"{safe_pct:>7.1f} {viab_pct:>7.1f}")

In [ ]:
# ===== Figure 4: Monte Carlo distributions + safety threshold =====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 5), facecolor='white')

window = df_mc_full[(df_mc_full['alpha']>=0.5) & (df_mc_full['alpha']<=0.7)]
for chassis in ['WT', 'KO11', 'LY01']:
    sub = window[window['chassis']==chassis]
    if len(sub) > 5:
        ax1.hist(sub['EtOH'], bins=18, color=COL[chassis], alpha=0.55,
                 edgecolor=COL[chassis], linewidth=0.8,
                 label=f'{chassis} (N={len(sub)})')

ax1.set_title('(a) Monte Carlo distribution in operating window',
              fontsize=11, fontweight='bold', loc='left', pad=8)
ax1.set_xlabel('Final ethanol titer (mM)', fontsize=10)
ax1.set_ylabel('Count', fontsize=10)
ax1.legend(fontsize=9.5, framealpha=0.92, edgecolor='#333')
ax1.grid(True, alpha=0.3, linestyle='--', lw=0.5)

for chassis in ['WT', 'KO11', 'LY01']:
    sub = df_sweep[(df_sweep['gsmm']=='iJO1366') & (df_sweep['chassis']==chassis)].sort_values('alpha')
    ax2.plot(sub['alpha'], sub['peak_S'], '-o', color=COL[chassis], lw=2, ms=5,
             markeredgecolor='white', markeredgewidth=0.5, label=chassis)

ax2.axhline(10.0, color='#d62728', ls='--', lw=1.5, alpha=0.8)
ax2.text(1.45, 10.3, 'Safety threshold (10 mM-eq)', fontsize=8.5, color='#d62728', ha='right')
ax2.fill_between([0, 1.55], 10, 18, color='#d62728', alpha=0.07)
ax2.axvspan(0.4, 0.7, alpha=0.12, color='#ffd700')
ax2.text(0.55, 0.5, 'Operating\nwindow', ha='center', fontsize=8.5, color='#806000', fontweight='bold')

ax2.set_title('(b) Stress safety threshold compliance',
              fontsize=11, fontweight='bold', loc='left', pad=8)
ax2.set_xlabel(r'Dimensionless PAW dose ($\alpha_{PAW}$)', fontsize=10)
ax2.set_ylabel('Peak stress $S_{peak}$ (mM-eq)', fontsize=10)
ax2.legend(fontsize=9.5, loc='upper left', framealpha=0.92, edgecolor='#333')
ax2.grid(True, alpha=0.3, linestyle='--', lw=0.5)
ax2.set_xlim(-0.05, 1.55)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'Figure_4_monte_carlo_safety.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: Figure_4_monte_carlo_safety.png")

---

## 13. Analysis 3.7 — Morris elementary-effects screening (§3.7)

In [ ]:
def morris_screening(model, chassis, n_traj=6, seed=123):
    keys = list(mc_ranges.keys())
    n_params = len(keys)
    rng = np.random.default_rng(seed)

    elem_effects = {k: [] for k in keys}

    for traj in tqdm(range(n_traj), desc=f'Morris {chassis}', leave=False):
        base_norm = rng.uniform(0, 1, n_params)
        delta = 0.5
        for i, k in enumerate(keys):
            perturbed_norm = base_norm.copy()
            perturbed_norm[i] = min(1.0, perturbed_norm[i] + delta)

            y_base, y_pert = 0, 0
            for params_norm, label in [(base_norm, 'base'), (perturbed_norm, 'pert')]:
                params = {}
                for k2, v_norm in zip(keys, params_norm):
                    lo, hi = mc_ranges[k2]
                    params[k2] = lo + v_norm * (hi - lo)

                orig = {k2: getattr(P, k2) for k2 in keys if k2 != 'alpha'}
                for k2, v in params.items():
                    if k2 != 'alpha':
                        setattr(P, k2, v)

                try:
                    res = run_dfba(model, chassis, params['alpha'])
                    if label == 'base':
                        y_base = res['EtOH_final']
                    else:
                        y_pert = res['EtOH_final']
                except Exception:
                    pass
                finally:
                    for k2, v in orig.items():
                        setattr(P, k2, v)

            ee = (y_pert - y_base) / delta
            elem_effects[k].append(ee)

    results = []
    for k in keys:
        ees = np.array(elem_effects[k])
        mu_star = np.abs(ees).mean()
        sigma = ees.std()
        results.append({'param': k, 'mu_star': mu_star, 'sigma': sigma,
                       'sigma_ratio': sigma/mu_star if mu_star > 0 else 0})

    return pd.DataFrame(results).sort_values('mu_star', ascending=False)

print("Morris screening (WT/iJO1366)...")
df_morris = morris_screening(m_J, 'WT', n_traj=6)
print()
print(df_morris.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

df_morris.to_csv(os.path.join(OUTPUT_DIR, 'morris_sensitivity.csv'), index=False)

---

## 14. Analysis 3.8 — Literature cross-validation (§3.8)

Reproduces **Table 4** with 14 anchors.

In [ ]:
anchors = [
    ('EXP-1',  'Anaerobic mu (WT)',                  0.242, 0.24,   'h^-1',         'Varma & Palsson (1994)'),
    ('EXP-2',  'PAW bulk H2O2 at alpha = 1',         5.0,   5.0,    'mM',           'Tarabova et al. (2018)'),
    ('EXP-3',  'H2O2 aqueous t_half',                 2.5,   2.5,    'h',            'Lukes et al. (2014)'),
    ('EXP-4',  'Downstream metabolic threshold',     1.8,   1.5,    'mM-eq',        'Park et al. (2005)'),
    ('EXP-5',  'WT EtOH yield',                      0.82,  0.84,   'mol/mol',      'Trinh et al. (2008)'),
    ('EXP-6',  'KO11 EtOH yield',                    0.84,  1.0,    'mol/mol',      'Ingram et al. (1991)'),
    ('EXP-7',  'LY01 EtOH yield',                    1.65,  1.7,    'mol/mol',      'Yomano et al. (1998)'),
    ('EXP-8',  'H2O2 bactericidal calibration',      14.0,  10.0,   'mM',           'Imlay (2008)'),
    ('EXP-9',  'PFL inactivation constant',          0.8,   0.9,    'mM (proxy)',   'Knappe & Sawers (1990)'),
    ('EXP-10', 'PAW dose-response peak alpha*',      0.60,  0.55,   'alpha_PAW',    'Internal concordance'),
    ('EXP-11', 'WT EtOH titer (10 h)',               17.0,  18.0,   'mM',           'Trinh et al. (2008)'),
    ('EXP-12', 'LY01 EtOH titer (10 h)',             25.2,  28.0,   'mM',           'Yomano et al. (1998)'),
    ('EXP-13', 'WT acetate (anaerobic)',             15.5,  16.0,   'mM',           'Forster et al. (2003)'),
    ('EXP-14', 'WT formate (anaerobic)',             36.0,  32.0,   'mM',           'Forster et al. (2003)'),
]

anchor_records = []
for anc_id, desc, pred, ref, units, source in anchors:
    dev_pct = (pred - ref) / ref * 100 if ref != 0 else 0
    abs_dev = abs(dev_pct)
    if abs_dev < 25:
        category = 'GOOD'
    elif abs_dev < 50:
        category = 'ACCEPTABLE'
    else:
        category = 'POOR'
    anchor_records.append({
        'ID': anc_id, 'Description': desc, 'Predicted': pred, 'Reference': ref,
        'Units': units, 'Source': source,
        'Deviation (%)': round(dev_pct, 1), 'Category': category
    })

df_anchors = pd.DataFrame(anchor_records)
print(df_anchors.to_string(index=False))
print()
print(f"GOOD:       {(df_anchors['Category']=='GOOD').sum()} / {len(df_anchors)}")
print(f"ACCEPTABLE: {(df_anchors['Category']=='ACCEPTABLE').sum()} / {len(df_anchors)}")
print(f"POOR:       {(df_anchors['Category']=='POOR').sum()} / {len(df_anchors)}")

df_anchors.to_csv(os.path.join(OUTPUT_DIR, 'table4_literature_anchors.csv'), index=False)

In [ ]:
# ===== Figure 5: Cross-GSMM concordance + literature anchors =====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), facecolor='white')

chassis_list = ['WT', 'KO11', 'LY01']
x_pos = np.arange(len(chassis_list))
width = 0.36

folds_J, folds_M = [], []
for c in chassis_list:
    sub_J = df_sweep[(df_sweep['gsmm']=='iJO1366') & (df_sweep['chassis']==c)]
    sub_M = df_sweep[(df_sweep['gsmm']=='iML1515') & (df_sweep['chassis']==c)]
    base_J = sub_J[sub_J['alpha']==0]['EtOH'].iloc[0]
    base_M = sub_M[sub_M['alpha']==0]['EtOH'].iloc[0]
    peak_J = sub_J['EtOH'].max()
    peak_M = sub_M['EtOH'].max()
    folds_J.append(peak_J / max(base_J, 0.01))
    folds_M.append(peak_M / max(base_M, 0.01))

ax1.bar(x_pos - width/2, folds_J, width, color='#1f77b4',
        edgecolor='#333', linewidth=0.8, label='iJO1366')
ax1.bar(x_pos + width/2, folds_M, width, color='#d62728',
        edgecolor='#333', linewidth=0.8, label='iML1515')
ax1.axhline(1.0, color='#666', ls=':', lw=1)
for i, (fj, fm) in enumerate(zip(folds_J, folds_M)):
    ax1.text(i - width/2, fj+0.02, f'{fj:.2f}', ha='center', fontsize=9.5, color='#1f77b4', fontweight='bold')
    ax1.text(i + width/2, fm+0.02, f'{fm:.2f}', ha='center', fontsize=9.5, color='#d62728', fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(chassis_list)
ax1.set_title('(a) Cross-GSMM fold-change concordance', fontsize=11, fontweight='bold', loc='left', pad=8)
ax1.set_xlabel('Chassis', fontsize=10)
ax1.set_ylabel('Hormetic fold-change', fontsize=10)
ax1.set_ylim(0.85, 1.45)
ax1.legend(fontsize=10, loc='upper right', framealpha=0.92, edgecolor='#333')
ax1.grid(True, alpha=0.3, linestyle='--', lw=0.5)

colors_map = {'GOOD': '#2ca02c', 'ACCEPTABLE': '#ff7f0e', 'POOR': '#d62728'}
bar_colors = [colors_map[c] for c in df_anchors['Category']]
y_pos = np.arange(len(df_anchors))
ax2.barh(y_pos, df_anchors['Deviation (%)'], color=bar_colors,
         edgecolor='#333', linewidth=0.6, alpha=0.85)
ax2.axvspan(-25, 25, color='#2ca02c', alpha=0.10)
ax2.axvspan(-50, -25, color='#ff7f0e', alpha=0.10)
ax2.axvspan(25, 50, color='#ff7f0e', alpha=0.10)
ax2.axvline(0, color='#333', lw=0.8)
ax2.axvline(-25, color='#666', ls='--', lw=0.6, alpha=0.7)
ax2.axvline(25, color='#666', ls='--', lw=0.6, alpha=0.7)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(df_anchors['ID'], fontsize=8.5)
ax2.invert_yaxis()
ax2.set_xlim(-60, 60)
ax2.set_title('(b) Literature cross-validation (14 anchors)',
              fontsize=11, fontweight='bold', loc='left', pad=8)
ax2.set_xlabel('Deviation from reference (%)', fontsize=10)
ax2.grid(True, alpha=0.3, linestyle='--', lw=0.5)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=colors_map['GOOD'], edgecolor='#333', alpha=0.85, label='GOOD (|d| < 25%)'),
    Patch(facecolor=colors_map['ACCEPTABLE'], edgecolor='#333', alpha=0.85, label='ACCEPTABLE (25-50%)'),
]
ax2.legend(handles=legend_elements, fontsize=9, loc='lower right', framealpha=0.92, edgecolor='#333')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'Figure_5_concordance_anchors.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: Figure_5_concordance_anchors.png")

---

## 15. Consolidate results to Excel workbook

In [ ]:
excel_path = os.path.join(OUTPUT_DIR, 'PAW_Ethanol_Results_ALL.xlsx')

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_baseline.to_excel(writer, sheet_name='Table2_Baseline', index=False)
    df_sweep.to_excel(writer, sheet_name='Figure1_Sweep', index=False)
    df_scenarios.to_excel(writer, sheet_name='Table3_Scenarios', index=False)
    df_mc_full.to_excel(writer, sheet_name='Figure4_MonteCarlo', index=False)
    df_morris.to_excel(writer, sheet_name='Morris_Sensitivity', index=False)
    df_anchors.to_excel(writer, sheet_name='Table4_Anchors', index=False)

print(f"All results consolidated into:\n  {excel_path}\n")
print("Contents:")
print("  - Table2_Baseline        - Cross-model baseline (Section 3.1)")
print("  - Figure1_Sweep          - Dose-response sweep (Section 3.2)")
print("  - Table3_Scenarios       - Scenario decomposition (Section 3.4)")
print("  - Figure4_MonteCarlo     - Monte Carlo samples (Section 3.6)")
print("  - Morris_Sensitivity     - Elementary effects ranking (Section 3.7)")
print("  - Table4_Anchors         - Literature cross-validation (Section 3.8)")

---

## 16. References

The 27 primary references are listed in the **References** section of the accompanying manuscript. Key citations relevant to this notebook:

**dFBA methodology**
- Mahadevan, R., Edwards, J. S., & Doyle, F. J. (2002). *Biophysical Journal*, 83(3), 1331-1340.
- Ebrahim, A. et al. (2013). COBRApy. *BMC Systems Biology*, 7, 74.

**Genome-scale reconstructions**
- Orth, J. D. et al. (2011). *Mol. Syst. Biol.*, 7, 535.
- Monk, J. M. et al. (2017). iML1515. *Nat. Biotechnol.*, 35(10), 904-908.

**Engineered chassis**
- Ohta, K. et al. (1991). *Appl. Env. Microbiol.*, 57(4), 893-900. *(KO11)*
- Yomano, L. P. et al. (1998). *J. Ind. Microbiol. Biotechnol.*, 20(2), 132-138. *(LY01)*

**PAW chemistry**
- Lukes, P. et al. (2014). *Plasma Sources Sci. Technol.*, 23(1), 015019.
- Tarabova, B. et al. (2018). *Plasma Proc. Polymers*, 15(6), e1800030.

**Oxidative stress biology**
- Imlay, J. A. (2008). *Annu. Rev. Biochem.*, 77, 755-776.
- Knappe, J., & Sawers, G. (1990). *FEMS Microbiol. Rev.*, 6(4), 383-398.

**Sensitivity analysis**
- McKay, M. D. et al. (1979). Latin hypercube sampling. *Technometrics*, 21(2), 239-245.
- Morris, M. D. (1991). *Technometrics*, 33(2), 161-174.

---

## License

This notebook and accompanying code are released under the **MIT License**. See `LICENSE` in the repository root.

## Contact

For questions, issues, or collaboration: open an issue at the repository, or contact the corresponding author of the manuscript.